# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/fatima12aa/fa-ml/blob/main/work/notebooks/w05_model.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

In [9]:
# ## Method choice and why
# (asked claude to ask me questions, i answered and then asked it to format and refine the answers. these answers are pasted below)
# We use two methods: **Decision Tree** and **Random Forest**, both classification methods
# (our target, `is_declining`, is a 0/1 label — not a continuous number, so Linear Regression
# is excluded; Logistic Regression, despite its name, IS a valid classification method, but we
# chose tree-based methods for the specific reason below).

# **Decision Tree** is chosen for its transparency: it can be printed as literal if/else rules,
# which matters directly for our lane — reviewers using our ranked queue need to trust *why* a
# page was flagged (same principle behind our w04 reason codes). A model whose logic can be
# read and verified builds more trust than an opaque score.

# **Random Forest** is included as a comparison, since it typically performs better (more
# predictive power, per notebook 02's reference numbers) but sacrifices that direct readability
# — it averages many trees together rather than offering one traceable path. Comparing both
# lets us see the real tradeoff between interpretability and performance, and choose deliberately
# rather than assuming "more complex = better."

SyntaxError: invalid character '—' (U+2014) (3069916010.py, line 4)

## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

In [ ]:
## Split design

We use a **grouped split by `client_hash_id`**, not a plain random split. Recall from w03's
lane guide: "pages from the same client may share patterns the model could memorize" — if the
same client's pages appeared in both train and test, the model could learn client-specific
quirks (a particular site's style, industry norms) rather than general decline patterns,
inflating our test score without real generalization. This matches the starter pipeline's own
approach (`split_strategy: client_holdout`, from notebook 01).

We use `GroupShuffleSplit` from scikit-learn, grouping on `client_hash_id`, so entire clients
are held out for testing — never split across train and test.

In [ ]:
# --- 1. Token + connection ---
import os
from google.colab import userdata
import duckdb

hf_token = userdata.get('HF_TOKEN')
os.environ["HF_TOKEN"] = hf_token

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{hf_token}')")

REL = 'hf://datasets/FlyRank/internship-warehouse'
TABLES = {
    'dim_clients':       f"read_parquet('{REL}/dim_clients.parquet')",
    'dim_content':       f"read_parquet('{REL}/dim_content.parquet')",
    'fact_daily':        f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')",
    'fact_daily_sample': f"read_parquet('{REL}/fact_content_daily_performance_sample.parquet')",
    'fact_query_90d':    f"read_parquet('{REL}/fact_content_query_90d.parquet')",
}
print("Connected.")

In [ ]:
# --- 2. February features (now including client_hash_id) ---
feb_impressions = con.sql(f"""
    SELECT content_hash_id, client_hash_id, SUM(gsc_impressions) AS feb_impressions
    FROM {TABLES['fact_daily']}
    WHERE report_date >= '2026-02-01' AND report_date < '2026-03-01'
      AND gsc_data_available IS TRUE
    GROUP BY content_hash_id, client_hash_id
""").df()

feb_ctr = con.sql(f"""
    SELECT content_hash_id,
        SUM(gsc_clicks) AS feb_clicks,
        SUM(gsc_impressions) AS feb_impressions_ctr,
        CASE WHEN SUM(gsc_impressions) > 0
             THEN CAST(SUM(gsc_clicks) AS DOUBLE) / SUM(gsc_impressions)
             ELSE NULL END AS feb_ctr
    FROM {TABLES['fact_daily']}
    WHERE report_date >= '2026-02-01' AND report_date < '2026-03-01'
      AND gsc_data_available IS TRUE
    GROUP BY content_hash_id
""").df()

feb_position = con.sql(f"""
    SELECT content_hash_id, AVG(gsc_avg_position) AS feb_avg_position
    FROM {TABLES['fact_daily']}
    WHERE report_date >= '2026-02-01' AND report_date < '2026-03-01'
      AND gsc_data_available IS TRUE AND gsc_avg_position > 0
    GROUP BY content_hash_id
""").df()

content_features = con.sql(f"""
    SELECT content_hash_id, word_count, search_volume
    FROM {TABLES['dim_content']}
""").df()

print("feb_impressions:", feb_impressions.shape)
print("feb_ctr:", feb_ctr.shape)
print("feb_position:", feb_position.shape)
print("content_features:", content_features.shape)

In [ ]:
# --- 3. March impressions + label ---
march_impressions = con.sql(f"""
    SELECT content_hash_id, SUM(gsc_impressions) AS march_impressions
    FROM {TABLES['fact_daily']}
    WHERE report_date >= '2026-03-01' AND report_date < '2026-04-01'
      AND gsc_data_available IS TRUE
    GROUP BY content_hash_id
""").df()

trend_data = feb_impressions.merge(march_impressions, on="content_hash_id", how="inner")
trend_data["is_declining"] = (
    trend_data["march_impressions"] < 0.8 * trend_data["feb_impressions"]
).astype(int)

print("trend_data:", trend_data.shape)

In [10]:
# --- 4. Assemble final training table WITH client_hash_id ---
training_table = content_features \
    .merge(feb_impressions, on="content_hash_id", how="inner") \
    .merge(feb_ctr[["content_hash_id", "feb_ctr"]], on="content_hash_id", how="left") \
    .merge(feb_position, on="content_hash_id", how="left") \
    .merge(trend_data[["content_hash_id", "is_declining"]], on="content_hash_id", how="inner")

feature_cols = ["word_count", "search_volume", "feb_impressions", "feb_ctr", "feb_avg_position"]
training_table_clean = training_table.dropna(subset=feature_cols)

print("training_table_clean:", training_table_clean.shape)
print(training_table_clean.columns.tolist())

training_table_clean: (75189, 8)
['content_hash_id', 'word_count', 'search_volume', 'client_hash_id', 'feb_impressions', 'feb_ctr', 'feb_avg_position', 'is_declining']


In [11]:
from sklearn.model_selection import GroupShuffleSplit

# We need client_hash_id in our training table to group by it.
# Let's check it's still there from your feature-building steps.
print(training_table_clean.columns.tolist())

['content_hash_id', 'word_count', 'search_volume', 'client_hash_id', 'feb_impressions', 'feb_ctr', 'feb_avg_position', 'is_declining']


In [12]:
from sklearn.model_selection import GroupShuffleSplit

feature_cols = ["word_count", "search_volume", "feb_impressions", "feb_ctr", "feb_avg_position"]
X = training_table_clean[feature_cols]
y = training_table_clean["is_declining"]
groups = training_table_clean["client_hash_id"]

gss = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42)
train_idx, test_idx = next(gss.split(X, y, groups=groups))

X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

# Verify: no client appears in both sets
train_clients = set(training_table_clean.iloc[train_idx]["client_hash_id"])
test_clients = set(training_table_clean.iloc[test_idx]["client_hash_id"])
overlap = train_clients & test_clients

print("Train rows:", len(X_train), "| Test rows:", len(X_test))
print("Train clients:", len(train_clients), "| Test clients:", len(test_clients))
print("Overlapping clients (should be 0):", len(overlap))

Train rows: 52791 | Test rows: 22398
Train clients: 28 | Test clients: 10
Overlapping clients (should be 0): 0


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.